# Week06 HW03 — Tree Ensemble to Boosting

이번 과제의 목표는 같은 tabular 데이터셋에서 **Random Forest, XGBoost, LightGBM**을 비교하고,  
Boosting 계열 모델의 **튜닝 포인트와 Early Stopping의 효과**를 확인하는 것이다.

본 실험에서는 Kaggle Titanic 데이터셋을 사용하여 다음 질문에 답하고자 한다.

1. Random Forest와 Boosting 계열은 어떤 차이를 보이는가?
2. XGBoost와 LightGBM은 어떤 튜닝 포인트를 가지는가?
3. 왜 train 점수보다 validation 구조가 더 중요한가?
4. Early Stopping은 어떤 효과를 가지는가?

## 1. 실험 환경 및 라이브러리 불러오기

먼저 실험에 필요한 라이브러리를 불러오고, 경고 메시지를 정리한다.
또한 재현 가능한 결과를 위해 random seed를 고정한다.

In [31]:
# 경고 메시지를 숨겨서 출력 결과를 깔끔하게 유지
import warnings
warnings.filterwarnings("ignore")

# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 경로 처리를 위한 라이브러리
from pathlib import Path

# 데이터 분할 및 전처리
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# 평가 지표
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 모델
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 재현성을 위한 시드 고정
RANDOM_STATE = 42

## 2. 데이터 경로 설정 및 데이터 불러오기

이번 과제에서는 Kaggle Titanic 데이터셋을 사용한다.  
평가의 핵심은 Kaggle 제출 점수보다 **내부 validation 구조를 통한 모델 비교**이므로,  
`train.csv`를 기준으로 train / validation / test를 다시 분할하여 실험을 진행한다.

참고로 `test.csv`는 Kaggle 제출용 파일이지만, 이번 과제의 핵심 비교에는 직접 사용하지 않는다.


In [32]:
# 실행 위치에 따라 경로가 달라질 수 있으므로 base directory를 유연하게 탐색
cwd = Path.cwd()

if (cwd / "data" / "raw" / "titanic").exists():
    BASE_DIR = cwd
elif (cwd.parent / "data" / "raw" / "titanic").exists():
    BASE_DIR = cwd.parent
else:
    BASE_DIR = cwd.parent  # 기본값
    print("Warning: titanic data directory was not found automatically. Check folder structure.")

# 데이터 및 출력 경로 설정
DATA_DIR = BASE_DIR / "data" / "raw" / "titanic"
OUTPUT_DIR = BASE_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
METRIC_DIR = OUTPUT_DIR / "metrics"

# 결과 저장 폴더가 없으면 생성
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", cwd)
print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())


Current working directory: c:\Users\qwer8\Dev\School\2-2_AI-Proj\Assignment3\notebooks
BASE_DIR: c:\Users\qwer8\Dev\School\2-2_AI-Proj\Assignment3
DATA_DIR: c:\Users\qwer8\Dev\School\2-2_AI-Proj\Assignment3\data\raw\titanic
DATA_DIR exists: True


In [33]:
# Titanic 학습 데이터와 Kaggle test 데이터를 불러옴
assert (DATA_DIR / "train.csv").exists(), "train.csv not found. Check Assignment3/data/raw/titanic/"
assert (DATA_DIR / "test.csv").exists(), "test.csv not found. Check Assignment3/data/raw/titanic/"

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df_kaggle = pd.read_csv(DATA_DIR / "test.csv")

# 데이터 크기와 상위 몇 개 행을 확인
print("train_df shape:", train_df.shape)
print("test_df_kaggle shape:", test_df_kaggle.shape)
display(train_df.head())


train_df shape: (891, 12)
test_df_kaggle shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. 데이터셋 소개 및 기본 확인

Titanic 데이터셋은 승객 정보를 바탕으로 생존 여부를 예측하는 **이진 분류(binary classification)** 문제이다.

- **문제 유형:** Classification
- **Target 변수:** `Survived`
- **주요 특징:** 수치형 + 범주형 혼합 데이터, 결측치 존재

In [34]:
# 데이터 타입, 결측치, 기본 정보 확인
print(train_df.info())

print("\n[Missing Values]")
print(train_df.isnull().sum().sort_values(ascending=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None

[Missing Values]
Cabin          687
Age            177
Embarked         2
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
SibSp            0
Parch            0
Ticket           0
Fare        

In [35]:
# target 분포 확인
train_df["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

## 4. Feature / Target 정의

이번 실험에서는 `Survived`를 target으로 사용한다.

또한 아래 컬럼은 이번 실험에서 제외한다.

- `PassengerId`: 식별자
- `Name`: 텍스트 기반 식별 정보
- `Ticket`: 고유값이 많아 단순 baseline 실험에서는 활용이 어려움
- `Cabin`: 결측치가 매우 많음

나머지 컬럼을 feature로 사용한다.

In [36]:
# 타깃 변수 정의
target_col = "Survived"

# 이번 실험에서 제외할 컬럼 정의
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]

# 사용할 feature 컬럼 목록 생성
feature_cols = [col for col in train_df.columns if col not in drop_cols + [target_col]]

# 입력(X)과 정답(y) 분리
X = train_df[feature_cols].copy()
y = train_df[target_col].copy()

print("Feature columns:", feature_cols)
print("Target column:", target_col)

Feature columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Target column: Survived


## 5. Train / Validation / Test 분할

이번 과제에서는 **train 성능보다 validation 구조를 더 중요하게 보기 위해**  
원본 `train.csv`를 다시 분할하여 다음과 같이 사용한다.

- Train: 모델 학습
- Validation: 하이퍼파라미터 비교 및 Early Stopping 기준
- Test: 최종 일반화 성능 확인

분류 문제이므로 target 비율이 유지되도록 `stratify` 옵션을 사용한다.

In [37]:
# 먼저 전체 데이터를 train_full / test로 분할
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

# train_full을 다시 train / validation으로 분할
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_STATE
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape :", X_test.shape)

X_train shape: (569, 7)
X_valid shape: (143, 7)
X_test shape : (179, 7)


## 6. 전처리 파이프라인 구성

Titanic 데이터는 수치형과 범주형 변수가 함께 존재하므로, 각 변수 타입에 맞는 전처리를 적용한다.

- **수치형 변수:** median으로 결측치 대체
- **범주형 변수:** 최빈값으로 결측치 대체 후 one-hot encoding

Tree 계열 모델은 스케일링에 덜 민감하므로, 이번 실험에서는 별도의 scaling은 적용하지 않는다.

In [38]:
# 수치형 변수와 범주형 변수 목록 지정
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]

# 수치형 전처리: 결측치를 median으로 대체
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# 범주형 전처리: 결측치를 최빈값으로 대체 후 one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# 컬럼별 전처리를 하나의 ColumnTransformer로 결합
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Preprocessor is ready.")

Preprocessor is ready.


## 7. 공통 평가 함수 정의

세 가지 모델(Random Forest, XGBoost, LightGBM)을 동일한 기준으로 비교하기 위해  
accuracy, F1, ROC-AUC를 계산하는 평가 함수를 정의한다.

이번 과제에서 중요한 점은 **train 점수뿐 아니라 validation / test 점수를 함께 보는 것**이다.

In [39]:
# 모델을 학습하고 train / validation / test 성능을 한 번에 계산하는 함수
def evaluate_model(model, X_train, y_train, X_valid, y_valid, X_test, y_test, model_name="model"):
    # 모델 학습
    model.fit(X_train, y_train)

    # 예측값 생성
    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    # 확률 예측값 생성 (ROC-AUC 계산용)
    train_prob = model.predict_proba(X_train)[:, 1]
    valid_prob = model.predict_proba(X_valid)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    # 결과를 dictionary 형태로 정리
    result = {
        "model": model_name,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "train_f1": f1_score(y_train, train_pred),
        "valid_f1": f1_score(y_valid, valid_pred),
        "test_f1": f1_score(y_test, test_pred),
        "train_roc_auc": roc_auc_score(y_train, train_prob),
        "valid_roc_auc": roc_auc_score(y_valid, valid_prob),
        "test_roc_auc": roc_auc_score(y_test, test_prob),
    }

    return result

## 8. Baseline 1 — Random Forest

먼저 Bagging 계열의 대표 모델인 Random Forest를 baseline으로 사용한다.

Random Forest는 여러 개의 트리를 병렬로 학습하여 분산(variance)을 줄이는 데 강점이 있으며,  
안정적인 baseline 모델로 자주 사용된다.

In [40]:
# Random Forest 파이프라인 구성
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=RANDOM_STATE
    ))
])

# Random Forest 성능 평가
rf_result = evaluate_model(
    rf_model,
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    model_name="RandomForest"
)

rf_result

{'model': 'RandomForest',
 'train_accuracy': 0.9806678383128296,
 'valid_accuracy': 0.8391608391608392,
 'test_accuracy': 0.770949720670391,
 'train_f1': 0.9748283752860412,
 'valid_f1': 0.780952380952381,
 'test_f1': 0.7050359712230215,
 'train_roc_auc': 0.9987649964714185,
 'valid_roc_auc': 0.8439049586776861,
 'test_roc_auc': 0.8207509881422925}

## 9. Baseline 2 — XGBoost

다음으로 Boosting 계열의 대표 모델인 XGBoost를 사용한다.

XGBoost는 이전 단계의 오류를 다음 단계가 순차적으로 보완하는 구조를 가지며,  
강력한 성능과 함께 정규화 및 실용적인 결측치 처리가 장점으로 알려져 있다.

In [41]:
# XGBoost 파이프라인 구성
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE
    ))
])

# XGBoost 성능 평가
xgb_result = evaluate_model(
    xgb_model,
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    model_name="XGBoost"
)

xgb_result

{'model': 'XGBoost',
 'train_accuracy': 0.9244288224956063,
 'valid_accuracy': 0.8531468531468531,
 'test_accuracy': 0.8100558659217877,
 'train_f1': 0.8958837772397095,
 'valid_f1': 0.8037383177570093,
 'test_f1': 0.734375,
 'train_roc_auc': 0.9797890692386105,
 'valid_roc_auc': 0.8815082644628098,
 'test_roc_auc': 0.8158761528326746}

## 10. Baseline 3 — LightGBM

마지막으로 LightGBM을 사용한다.

LightGBM은 histogram 기반 학습과 leaf-wise 성장 전략을 사용하며,  
속도와 메모리 효율 측면에서 강점을 가지는 Boosting 모델이다.

In [42]:
# LightGBM 파이프라인 구성
lgbm_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=RANDOM_STATE
    ))
])

# LightGBM 성능 평가
lgbm_result = evaluate_model(
    lgbm_model,
    X_train, y_train,
    X_valid, y_valid,
    X_test, y_test,
    model_name="LightGBM"
)

lgbm_result

[LightGBM] [Info] Number of positive: 218, number of negative: 351
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001872 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 190
[LightGBM] [Info] Number of data points in the train set: 569, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383128 -> initscore=-0.476291
[LightGBM] [Info] Start training from score -0.476291
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

{'model': 'LightGBM',
 'train_accuracy': 0.9630931458699473,
 'valid_accuracy': 0.8321678321678322,
 'test_accuracy': 0.7932960893854749,
 'train_f1': 0.9505882352941176,
 'valid_f1': 0.7692307692307693,
 'test_f1': 0.7175572519083969,
 'train_roc_auc': 0.9960336130060901,
 'valid_roc_auc': 0.8668388429752066,
 'test_roc_auc': 0.8232542819499341}

## 11. 1차 모델 비교표

세 가지 모델의 성능을 한 표로 정리한다.

이 단계에서는 **train 점수보다 validation 점수와 test 점수의 차이**,  
그리고 **어떤 모델이 더 안정적으로 일반화되는지**를 확인하는 것이 중요하다.

In [43]:
# baseline 결과를 하나의 DataFrame으로 정리
results_df = pd.DataFrame([rf_result, xgb_result, lgbm_result])

# validation accuracy 기준으로 정렬
results_df = results_df.sort_values(by="valid_accuracy", ascending=False).reset_index(drop=True)

results_df

,model,train_accuracy,valid_accuracy,test_accuracy,train_f1,valid_f1,test_f1,train_roc_auc,valid_roc_auc,test_roc_auc
0,XGBoost,0.924429,0.853147,0.810056,0.895884,0.803738,0.734375,0.979789,0.881508,0.815876
1,RandomForest,0.980668,0.839161,0.770950,0.974828,0.780952,0.705036,0.998765,0.843905,0.820751
2,LightGBM,0.963093,0.832168,0.793296,0.950588,0.769231,0.717557,0.996034,0.866839,0.823254


## 12. 1차 결과 해석

1차 비교 결과를 통해 다음을 확인한다.

- Random Forest는 안정적인 baseline 역할을 하는가?
- Boosting 계열(XGBoost, LightGBM)이 validation 기준으로 더 우수한가?
- train 점수와 validation 점수 차이로 볼 때 과적합 신호가 있는가?

이 결과를 바탕으로 다음 단계에서는 XGBoost를 중심으로 하이퍼파라미터 튜닝과 Early Stopping을 적용한다.

## 13. Boosting 튜닝을 위한 전처리 데이터 생성

Early Stopping을 적용하려면 validation set을 별도의 `eval_set`으로 전달해야 한다.  
이를 위해 전처리 파이프라인을 **train 데이터에만 다시 학습(fit)** 한 뒤,  
train / validation / test를 같은 기준으로 변환한다.

이 과정은 데이터 누수를 만들지 않으며, baseline 단계와 동일하게 **train 기준 전처리**를 유지한다.


In [44]:
# 전처리 파이프라인을 train 데이터에 맞춰 학습
X_train_enc = preprocessor.fit_transform(X_train)
X_valid_enc = preprocessor.transform(X_valid)
X_test_enc = preprocessor.transform(X_test)

print("X_train_enc shape:", X_train_enc.shape)
print("X_valid_enc shape:", X_valid_enc.shape)
print("X_test_enc shape :", X_test_enc.shape)

X_train_enc shape: (569, 12)
X_valid_enc shape: (143, 12)
X_test_enc shape : (179, 12)


## 14. XGBoost 튜닝용 평가 함수

이제 전처리가 완료된 데이터를 사용하여  
XGBoost의 하이퍼파라미터 조합별 성능을 비교한다.

In [45]:
# boosting 모델 튜닝 실험용 평가 함수
def evaluate_boosting_model(model, X_train, y_train, X_valid, y_valid, X_test, y_test, model_name="model"):
    # 모델 학습
    model.fit(X_train, y_train)

    # 예측값 생성
    train_pred = model.predict(X_train)
    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    # 확률 예측값 생성
    train_prob = model.predict_proba(X_train)[:, 1]
    valid_prob = model.predict_proba(X_valid)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    # 결과 정리
    return {
        "model": model_name,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "train_f1": f1_score(y_train, train_pred),
        "valid_f1": f1_score(y_valid, valid_pred),
        "test_f1": f1_score(y_test, test_pred),
        "train_roc_auc": roc_auc_score(y_train, train_prob),
        "valid_roc_auc": roc_auc_score(y_valid, valid_prob),
        "test_roc_auc": roc_auc_score(y_test, test_prob),
    }

## 15. XGBoost 하이퍼파라미터 튜닝

이번 실험에서는 최소 2개 이상의 핵심 하이퍼파라미터를 조정해야 하므로,  
다음 파라미터를 중심으로 비교한다.

- `learning_rate`
- `n_estimators`
- `max_depth`

각 조합에 대해 validation 성능을 비교하고, 가장 적절한 설정을 찾는다.

In [46]:
# 여러 하이퍼파라미터 조합을 저장할 리스트
xgb_experiments = []

# 비교할 파라미터 조합 정의
param_grid = [
    {"learning_rate": 0.03, "n_estimators": 300, "max_depth": 3},
    {"learning_rate": 0.05, "n_estimators": 300, "max_depth": 4},
    {"learning_rate": 0.05, "n_estimators": 500, "max_depth": 3},
    {"learning_rate": 0.10, "n_estimators": 300, "max_depth": 3},
    {"learning_rate": 0.05, "n_estimators": 500, "max_depth": 4},
]

# 각 조합을 순회하며 성능 평가
for i, params in enumerate(param_grid, start=1):
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        subsample=0.9,
        colsample_bytree=0.9,
        **params
    )

    result = evaluate_boosting_model(
        model,
        X_train_enc, y_train,
        X_valid_enc, y_valid,
        X_test_enc, y_test,
        model_name=f"XGB_tuned_{i}"
    )

    # 어떤 파라미터 조합이었는지 함께 저장
    result.update(params)
    xgb_experiments.append(result)

# 결과를 DataFrame으로 정리
xgb_tuning_df = pd.DataFrame(xgb_experiments).sort_values(
    by=["valid_accuracy", "valid_roc_auc"],
    ascending=False
).reset_index(drop=True)

xgb_tuning_df

,model,train_accuracy,valid_accuracy,test_accuracy,train_f1,valid_f1,test_f1,train_roc_auc,valid_roc_auc,test_roc_auc,learning_rate,n_estimators,max_depth
0,XGB_tuned_2,0.924429,0.853147,0.810056,0.895884,0.803738,0.734375,0.979789,0.881508,0.815876,0.05,300,4
1,XGB_tuned_3,0.924429,0.846154,0.815642,0.895884,0.792453,0.744186,0.977541,0.881715,0.811397,0.05,500,3
2,XGB_tuned_1,0.891037,0.832168,0.798883,0.846535,0.769231,0.714286,0.944562,0.887913,0.812451,0.03,300,3
3,XGB_tuned_4,0.927944,0.832168,0.804469,0.901679,0.773585,0.728682,0.982468,0.873244,0.813834,0.10,300,3
4,XGB_tuned_5,0.934974,0.825175,0.810056,0.912941,0.770642,0.738462,0.990545,0.872004,0.818643,0.05,500,4


In [47]:
# 보고서에 넣기 좋은 핵심 컬럼만 확인
xgb_tuning_df[
    [
        "model",
        "learning_rate",
        "n_estimators",
        "max_depth",
        "valid_accuracy",
        "valid_f1",
        "valid_roc_auc",
        "test_accuracy",
        "test_f1",
        "test_roc_auc"
    ]
]

,model,learning_rate,n_estimators,max_depth,valid_accuracy,valid_f1,valid_roc_auc,test_accuracy,test_f1,test_roc_auc
0,XGB_tuned_2,0.05,300,4,0.853147,0.803738,0.881508,0.810056,0.734375,0.815876
1,XGB_tuned_3,0.05,500,3,0.846154,0.792453,0.881715,0.815642,0.744186,0.811397
2,XGB_tuned_1,0.03,300,3,0.832168,0.769231,0.887913,0.798883,0.714286,0.812451
3,XGB_tuned_4,0.10,300,3,0.832168,0.773585,0.873244,0.804469,0.728682,0.813834
4,XGB_tuned_5,0.05,500,4,0.825175,0.770642,0.872004,0.810056,0.738462,0.818643


In [48]:
# validation 성능을 우선하고, 동률이면 ROC-AUC와 과적합 정도(acc gap)를 함께 고려해 최종 파라미터를 선택
xgb_tuning_df["acc_gap"] = xgb_tuning_df["train_accuracy"] - xgb_tuning_df["valid_accuracy"]
xgb_tuning_df["f1_gap"] = xgb_tuning_df["train_f1"] - xgb_tuning_df["valid_f1"]
xgb_tuning_df["auc_gap"] = xgb_tuning_df["train_roc_auc"] - xgb_tuning_df["valid_roc_auc"]

selection_df = xgb_tuning_df[
    [
        "model",
        "learning_rate",
        "n_estimators",
        "max_depth",
        "train_accuracy",
        "valid_accuracy",
        "valid_f1",
        "valid_roc_auc",
        "acc_gap",
    ]
].sort_values(
    by=["valid_accuracy", "valid_roc_auc", "acc_gap"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(selection_df)

best_row = selection_df.iloc[0]
best_params = {
    "learning_rate": float(best_row["learning_rate"]),
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"]),
}

print("Selected best_params:", best_params)


,model,learning_rate,n_estimators,max_depth,train_accuracy,valid_accuracy,valid_f1,valid_roc_auc,acc_gap
0,XGB_tuned_2,0.05,300,4,0.924429,0.853147,0.803738,0.881508,0.071282
1,XGB_tuned_3,0.05,500,3,0.924429,0.846154,0.792453,0.881715,0.078275
2,XGB_tuned_1,0.03,300,3,0.891037,0.832168,0.769231,0.887913,0.058869
3,XGB_tuned_4,0.10,300,3,0.927944,0.832168,0.773585,0.873244,0.095776
4,XGB_tuned_5,0.05,500,4,0.934974,0.825175,0.770642,0.872004,0.109799


Selected best_params: {'learning_rate': 0.05, 'n_estimators': 300, 'max_depth': 4}


## 16. 최적 파라미터 선택 기준 정리

이번 과제에서는 **train 점수보다 validation 성능**을 더 중요하게 본다.  
따라서 XGBoost의 최적 하이퍼파라미터는 아래 기준으로 선택한다.

1. `valid_accuracy`가 가장 높은 조합
2. 동률이면 `valid_roc_auc`가 더 높은 조합
3. 그래도 비슷하면 `acc_gap(train - valid)`이 더 작은 조합

즉, 단순히 train 성능이 가장 높은 모델이 아니라,  
**validation 성능이 좋고 과적합이 덜한 모델**을 최종 후보로 선택한다.


## 17. Early Stopping 적용 전 XGBoost

먼저 선택된 하이퍼파라미터로 **Early Stopping 없이** XGBoost를 학습한다.  
이 결과는 이후 Early Stopping 적용 모델과 비교하기 위한 기준점 역할을 한다.


In [49]:
# 선택된 파라미터로 Early Stopping 없이 XGBoost 학습
xgb_no_es = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    subsample=0.9,
    colsample_bytree=0.9,
    **best_params
)

xgb_no_es_result = evaluate_boosting_model(
    xgb_no_es,
    X_train_enc, y_train,
    X_valid_enc, y_valid,
    X_test_enc, y_test,
    model_name="XGBoost_no_early_stopping"
)

pd.DataFrame([xgb_no_es_result])

,model,train_accuracy,valid_accuracy,test_accuracy,train_f1,valid_f1,test_f1,train_roc_auc,valid_roc_auc,test_roc_auc
0,XGBoost_no_early_stopping,0.924429,0.853147,0.810056,0.895884,0.803738,0.734375,0.979789,0.881508,0.815876


## 18. Early Stopping 적용 XGBoost

이번에는 validation set을 기준으로 Early Stopping을 적용한다.  
validation 성능이 더 이상 개선되지 않으면 학습을 조기에 종료하여  
과적합을 줄이고 불필요한 boosting round를 방지한다.


In [50]:
# xgboost 버전에 따라 early stopping 인자 위치가 다를 수 있으므로 예외 처리 포함
try:
    xgb_es = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        subsample=0.9,
        colsample_bytree=0.9,
        early_stopping_rounds=30,
        **best_params
    )

    xgb_es.fit(
        X_train_enc, y_train,
        eval_set=[(X_valid_enc, y_valid)],
        verbose=False
    )
except TypeError:
    xgb_es = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        subsample=0.9,
        colsample_bytree=0.9,
        **best_params
    )

    xgb_es.fit(
        X_train_enc, y_train,
        eval_set=[(X_valid_enc, y_valid)],
        verbose=False,
        early_stopping_rounds=30
    )

# Early Stopping 적용 모델 예측값과 확률값 생성
xgb_es_train_pred = xgb_es.predict(X_train_enc)
xgb_es_valid_pred = xgb_es.predict(X_valid_enc)
xgb_es_test_pred = xgb_es.predict(X_test_enc)

xgb_es_train_prob = xgb_es.predict_proba(X_train_enc)[:, 1]
xgb_es_valid_prob = xgb_es.predict_proba(X_valid_enc)[:, 1]
xgb_es_test_prob = xgb_es.predict_proba(X_test_enc)[:, 1]

# 결과 정리
xgb_es_result = {
    "model": "XGBoost_with_early_stopping",
    "best_iteration": getattr(xgb_es, "best_iteration", None),
    "train_accuracy": accuracy_score(y_train, xgb_es_train_pred),
    "valid_accuracy": accuracy_score(y_valid, xgb_es_valid_pred),
    "test_accuracy": accuracy_score(y_test, xgb_es_test_pred),
    "train_f1": f1_score(y_train, xgb_es_train_pred),
    "valid_f1": f1_score(y_valid, xgb_es_valid_pred),
    "test_f1": f1_score(y_test, xgb_es_test_pred),
    "train_roc_auc": roc_auc_score(y_train, xgb_es_train_prob),
    "valid_roc_auc": roc_auc_score(y_valid, xgb_es_valid_prob),
    "test_roc_auc": roc_auc_score(y_test, xgb_es_test_prob),
}

pd.DataFrame([xgb_es_result])

,model,best_iteration,train_accuracy,valid_accuracy,test_accuracy,train_f1,valid_f1,test_f1,train_roc_auc,valid_roc_auc,test_roc_auc
0,XGBoost_with_early_stopping,150,0.903339,0.846154,0.815642,0.864865,0.792453,0.740157,0.95833,0.891839,0.811594


## 19. Early Stopping 적용 전후 비교

이제 Early Stopping 적용 전과 후의 성능을 비교한다.

관찰 포인트는 다음과 같다.

- validation 성능이 개선되었는가?
- train 성능이 과도하게 높아지는 현상이 줄어들었는가?
- test 성능이 유지되거나 개선되었는가?


In [51]:
# Early Stopping 적용 전후 결과 비교
es_compare_df = pd.DataFrame([xgb_no_es_result, xgb_es_result])
es_compare_df

,model,train_accuracy,valid_accuracy,test_accuracy,train_f1,valid_f1,test_f1,train_roc_auc,valid_roc_auc,test_roc_auc,best_iteration
0,XGBoost_no_early_stopping,0.924429,0.853147,0.810056,0.895884,0.803738,0.734375,0.979789,0.881508,0.815876,NaN
1,XGBoost_with_early_stopping,0.903339,0.846154,0.815642,0.864865,0.792453,0.740157,0.958330,0.891839,0.811594,150.0


## 20. 최종 모델 비교표

초기 baseline 결과와 XGBoost 튜닝 / Early Stopping 결과를 함께 정리하여  
최종적으로 어떤 모델을 선택할지 판단한다.


In [52]:
# baseline 결과와 XGBoost 추가 실험 결과를 합쳐 최종 비교표 생성
final_compare_df = pd.concat(
    [
        results_df,
        pd.DataFrame([xgb_no_es_result, xgb_es_result])
    ],
    ignore_index=True
)

# validation 성능 기준으로 다시 정렬
sort_cols = [col for col in ["valid_accuracy", "valid_roc_auc", "valid_f1"] if col in final_compare_df.columns]
final_compare_df = final_compare_df.sort_values(
    by=sort_cols,
    ascending=[False] * len(sort_cols)
).reset_index(drop=True)

final_compare_df

,model,train_accuracy,valid_accuracy,test_accuracy,train_f1,valid_f1,test_f1,train_roc_auc,valid_roc_auc,test_roc_auc,best_iteration
0,XGBoost,0.924429,0.853147,0.810056,0.895884,0.803738,0.734375,0.979789,0.881508,0.815876,NaN
1,XGBoost_no_early_stopping,0.924429,0.853147,0.810056,0.895884,0.803738,0.734375,0.979789,0.881508,0.815876,NaN
2,XGBoost_with_early_stopping,0.903339,0.846154,0.815642,0.864865,0.792453,0.740157,0.958330,0.891839,0.811594,150.0
3,RandomForest,0.980668,0.839161,0.770950,0.974828,0.780952,0.705036,0.998765,0.843905,0.820751,NaN
4,LightGBM,0.963093,0.832168,0.793296,0.950588,0.769231,0.717557,0.996034,0.866839,0.823254,NaN


## 21. 결과 저장

보고서 작성과 재현성을 위해 결과표를 csv 파일로 저장한다.


In [53]:
# 결과 저장
results_df.to_csv(METRIC_DIR / "baseline_results.csv", index=False)
xgb_tuning_df.to_csv(METRIC_DIR / "xgb_tuning_results.csv", index=False)
es_compare_df.to_csv(METRIC_DIR / "xgb_early_stopping_results.csv", index=False)
final_compare_df.to_csv(METRIC_DIR / "final_compare_results.csv", index=False)

print("All result tables have been saved successfully.")

All result tables have been saved successfully.


## 22. 결과 해석

이번 실험을 통해 다음 내용을 확인할 수 있다.

1. **Random Forest vs Boosting**
   - Random Forest는 안정적인 baseline 역할을 했지만, train 성능이 지나치게 높게 나오면 validation / test 성능과 차이가 커질 수 있었다.
   - 반면 Boosting 계열은 validation 기준에서 더 강한 성능을 보일 가능성이 높았고, 특히 XGBoost가 더 안정적인 후보로 나타났다.

2. **XGBoost vs LightGBM**
   - 두 모델 모두 강력한 tabular 분류기였지만, 이번 실험에서는 XGBoost가 validation 기준으로 더 일관된 성능을 보였다.
   - LightGBM도 경쟁력 있는 결과를 냈지만, 이번 설정에서는 최종 선택 모델로는 XGBoost가 더 적절했다.

3. **Validation의 중요성**
   - train 점수만 보면 모델이 매우 좋아 보일 수 있지만, 실제 모델 선택은 validation / test 성능을 기준으로 이루어져야 한다.
   - 특히 Boosting 계열은 성능이 강한 만큼 과적합 통제가 더 중요하므로, validation 구조가 필수적이다.

4. **Early Stopping의 효과**
   - validation 성능이 더 이상 개선되지 않는 시점에서 학습을 멈추어 과적합을 줄이는 데 도움이 된다.
   - 동시에 불필요한 boosting round를 줄여 학습 효율도 개선할 수 있다.

따라서 이번 과제의 최종 모델은 **validation 성능과 일반화 가능성**을 함께 고려하여 선택한다.


## 23. 제출용 요약

- **데이터셋:** Kaggle Titanic
- **문제 유형:** Binary Classification
- **Target:** `Survived`
- **비교 모델:** Random Forest, XGBoost, LightGBM
- **튜닝 모델:** XGBoost
- **조정한 핵심 하이퍼파라미터:** `learning_rate`, `n_estimators`, `max_depth`
- **Early Stopping 적용:** XGBoost에 적용
- **최종 선택 기준:** validation 성능과 test 일반화 성능을 함께 고려

이번 과제의 핵심은 단순 최고 점수보다,  
**Bagging 계열과 Boosting 계열의 차이**,  
**validation 기반 실험 설계의 중요성**,  
**Early Stopping을 통한 과적합 제어**를 설명하는 데 있다.
